# ML/AI Fundamentals Interview Prep

> **Goal**: Master the most commonly asked ML/AI interview questions with practical implementations
>
> **Coverage**:
> - Core ML concepts
> - Deep Learning fundamentals
> - System design questions
> - Coding challenges
> - Case studies
>
> **Companies**: Google, Meta, Amazon, Microsoft, OpenAI, Anthropic, DeepMind

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

## Section 1: Classic ML Interview Questions

### Q1: Bias-Variance Tradeoff

**Interviewer**: "Explain the bias-variance tradeoff. How does it relate to model complexity?"

**Answer Framework**:

1. **Definitions**:
   - **Bias**: Error from wrong assumptions (underfitting)
   - **Variance**: Error from sensitivity to training data (overfitting)
   - **Irreducible Error**: Noise inherent in data

2. **Mathematical Formulation**:
   $$\text{Expected Test Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

3. **Practical Implications**:
   - Simple models: High bias, low variance
   - Complex models: Low bias, high variance
   - Goal: Find the sweet spot

Let's demonstrate with code:

In [ ]:
# Generate data with noise
np.random.seed(42)
X = np.linspace(0, 10, 100)
y_true = 2 * X + 5
y = y_true + np.random.normal(0, 2, 100)  # Add noise

# Fit models of different complexities
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
complexities = [1, 5, 15]  # Polynomial degrees
titles = ['High Bias (Underfitting)', 'Good Balance', 'High Variance (Overfitting)']

for ax, degree, title in zip(axes, complexities, titles):
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=0.1))
    model.fit(X.reshape(-1, 1), y)
    
    X_plot = np.linspace(0, 10, 300)
    y_plot = model.predict(X_plot.reshape(-1, 1))
    
    ax.scatter(X, y, alpha=0.5, label='Data')
    ax.plot(X, y_true, 'g--', label='True Function', linewidth=2)
    ax.plot(X_plot, y_plot, 'r-', label=f'Model (degree={degree})', linewidth=2)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

print("Key Takeaway: Model complexity must match problem complexity!")

### Q2: Implement K-Fold Cross-Validation from Scratch

**Interviewer**: "Implement k-fold cross-validation without using sklearn."

In [ ]:
def k_fold_cross_validation(X, y, model_class, k=5, **model_kwargs):
    """
    Implement k-fold cross-validation from scratch.
    
    Args:
        X: np.array, features
        y: np.array, labels
        model_class: class with fit() and predict() methods
        k: int, number of folds
        **model_kwargs: arguments to pass to model_class
    
    Returns:
        fold_scores: list of scores for each fold
        mean_score: average score across folds
    """
    n_samples = len(X)
    fold_size = n_samples // k
    
    # Shuffle indices
    indices = np.random.permutation(n_samples)
    
    fold_scores = []
    
    for fold in range(k):
        # Split indices
        val_start = fold * fold_size
        val_end = val_start + fold_size if fold < k - 1 else n_samples
        
        val_indices = indices[val_start:val_end]
        train_indices = np.concatenate([indices[:val_start], indices[val_end:]])
        
        # Split data
        X_train, X_val = X[train_indices], X[val_indices]
        y_train, y_val = y[train_indices], y[val_indices]
        
        # Train and evaluate
        model = model_class(**model_kwargs)
        model.fit(X_train, y_train)
        score = model.score(X_val, y_val)
        
        fold_scores.append(score)
        print(f"Fold {fold + 1}/{k}: Score = {score:.4f}")
    
    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    
    print(f"\nMean Score: {mean_score:.4f} ± {std_score:.4f}")
    
    return fold_scores, mean_score

# Test implementation
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
scores, mean = k_fold_cross_validation(X, y, LogisticRegression, k=5, max_iter=1000)

### Q3: Gradient Descent Variants

**Interviewer**: "Explain and implement different gradient descent variants."

**Key Variants**:
1. **Batch Gradient Descent**: Uses all data
2. **Stochastic Gradient Descent**: Uses one sample
3. **Mini-Batch Gradient Descent**: Uses batch of samples
4. **Momentum**: Adds velocity term
5. **Adam**: Adaptive learning rate

In [ ]:
class GradientDescentOptimizers:
    """Collection of gradient descent variants."""
    
    @staticmethod
    def sgd(params, grads, lr=0.01):
        """Vanilla SGD."""
        return params - lr * grads
    
    @staticmethod
    def momentum(params, grads, velocity, lr=0.01, beta=0.9):
        """SGD with momentum.
        
        Momentum helps accelerate convergence and dampens oscillations.
        """
        velocity = beta * velocity + (1 - beta) * grads
        params = params - lr * velocity
        return params, velocity
    
    @staticmethod
    def adam(params, grads, m, v, t, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        """Adam optimizer.
        
        Combines momentum and RMSprop.
        
        Args:
            m: First moment estimate (momentum)
            v: Second moment estimate (RMSprop)
            t: Time step
        """
        # Update biased first moment estimate
        m = beta1 * m + (1 - beta1) * grads
        
        # Update biased second moment estimate
        v = beta2 * v + (1 - beta2) * (grads ** 2)
        
        # Bias correction
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        
        # Update parameters
        params = params - lr * m_hat / (np.sqrt(v_hat) + eps)
        
        return params, m, v

# Demonstration: Optimize a simple quadratic function
def f(x, y):
    """Objective function: x^2 + 10*y^2"""
    return x**2 + 10*y**2

def grad_f(x, y):
    """Gradient of objective."""
    return np.array([2*x, 20*y])

# Compare optimizers
def compare_optimizers(init_pos, steps=50):
    results = {}
    
    # SGD
    pos = init_pos.copy()
    trajectory_sgd = [pos.copy()]
    for _ in range(steps):
        grad = grad_f(*pos)
        pos = GradientDescentOptimizers.sgd(pos, grad, lr=0.1)
        trajectory_sgd.append(pos.copy())
    results['SGD'] = np.array(trajectory_sgd)
    
    # Momentum
    pos = init_pos.copy()
    velocity = np.zeros_like(pos)
    trajectory_momentum = [pos.copy()]
    for _ in range(steps):
        grad = grad_f(*pos)
        pos, velocity = GradientDescentOptimizers.momentum(pos, grad, velocity, lr=0.1)
        trajectory_momentum.append(pos.copy())
    results['Momentum'] = np.array(trajectory_momentum)
    
    # Adam
    pos = init_pos.copy()
    m, v = np.zeros_like(pos), np.zeros_like(pos)
    trajectory_adam = [pos.copy()]
    for t in range(1, steps + 1):
        grad = grad_f(*pos)
        pos, m, v = GradientDescentOptimizers.adam(pos, grad, m, v, t, lr=0.5)
        trajectory_adam.append(pos.copy())
    results['Adam'] = np.array(trajectory_adam)
    
    return results

# Visualize
init_pos = np.array([10.0, 10.0])
results = compare_optimizers(init_pos)

plt.figure(figsize=(12, 5))

# Plot contours
x = np.linspace(-12, 12, 100)
y = np.linspace(-12, 12, 100)
X, Y = np.meshgrid(x, y)
Z = f(X, Y)

plt.contour(X, Y, Z, levels=20, alpha=0.3)
plt.scatter([0], [0], c='red', s=100, marker='*', label='Optimum', zorder=10)

# Plot trajectories
colors = {'SGD': 'blue', 'Momentum': 'green', 'Adam': 'orange'}
for name, traj in results.items():
    plt.plot(traj[:, 0], traj[:, 1], 'o-', label=name, color=colors[name], alpha=0.7)

plt.xlabel('x')
plt.ylabel('y')
plt.title('Optimizer Comparison on Quadratic Function')
plt.legend()
plt.grid(True)
plt.show()

print("Observation: Adam converges fastest, Momentum smoother than SGD!")

## Section 2: Deep Learning Conceptual Questions

### Q4: Batch Normalization

**Interviewer**: "Explain batch normalization. Why is it useful? Implement it from scratch."

In [ ]:
class BatchNorm1D:
    """Batch Normalization for 1D inputs (MLPs).
    
    Benefits:
    1. Reduces internal covariate shift
    2. Allows higher learning rates
    3. Acts as regularization
    4. Makes network more stable
    
    Formula:
        y = gamma * (x - mean) / sqrt(var + eps) + beta
    """
    
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # Learnable parameters
        self.gamma = np.ones(num_features)   # Scale
        self.beta = np.zeros(num_features)   # Shift
        
        # Running statistics (for inference)
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)
    
    def forward(self, x, training=True):
        """Forward pass.
        
        Args:
            x: (batch_size, num_features)
            training: bool, use batch stats or running stats
        """
        if training:
            # Compute batch statistics
            batch_mean = np.mean(x, axis=0)
            batch_var = np.var(x, axis=0)
            
            # Update running statistics
            self.running_mean = (1 - self.momentum) * self.running_mean + \
                               self.momentum * batch_mean
            self.running_var = (1 - self.momentum) * self.running_var + \
                              self.momentum * batch_var
            
            # Normalize
            x_norm = (x - batch_mean) / np.sqrt(batch_var + self.eps)
        else:
            # Use running statistics
            x_norm = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
        
        # Scale and shift
        out = self.gamma * x_norm + self.beta
        
        return out

# Test BatchNorm
bn = BatchNorm1D(num_features=3)
x = np.random.randn(10, 3) * 5 + 10  # Mean ~10, Std ~5

print("Input statistics:")
print(f"  Mean: {x.mean(axis=0)}")
print(f"  Std:  {x.std(axis=0)}")

x_norm = bn.forward(x, training=True)

print("\nAfter BatchNorm:")
print(f"  Mean: {x_norm.mean(axis=0)}")
print(f"  Std:  {x_norm.std(axis=0)}")
print("\nAs expected: mean ~0, std ~1")

### Q5: Dropout

**Interviewer**: "How does dropout work? Implement it and explain why we scale during inference."

In [ ]:
class Dropout:
    """Dropout regularization.
    
    During training: Randomly zero out neurons with probability p
    During inference: Scale activations by (1-p)
    
    Why it works:
    - Prevents co-adaptation of neurons
    - Acts as ensemble learning
    - Adds noise → regularization
    """
    
    def __init__(self, keep_prob=0.8):
        self.keep_prob = keep_prob
    
    def forward(self, x, training=True):
        if training:
            # Create dropout mask
            mask = (np.random.rand(*x.shape) < self.keep_prob).astype(float)
            
            # Apply mask and scale
            # Scale by 1/keep_prob to maintain expected value
            out = (x * mask) / self.keep_prob
        else:
            # During inference: no dropout, no scaling needed
            out = x
        
        return out

# Demonstrate expected value preservation
dropout = Dropout(keep_prob=0.8)

x = np.ones((1000, 10)) * 5.0  # All values = 5

print(f"Input mean: {x.mean():.4f}")

# Training mode
x_train = dropout.forward(x, training=True)
print(f"After dropout (training): {x_train.mean():.4f}")
print(f"  (Should be close to 5.0 due to scaling)")

# Inference mode
x_infer = dropout.forward(x, training=False)
print(f"After dropout (inference): {x_infer.mean():.4f}")
print(f"  (Exactly 5.0 - no dropout applied)")

## Section 3: Coding Challenges

### Q6: Implement Softmax from Scratch

**Interviewer**: "Implement numerically stable softmax. Handle edge cases."

In [ ]:
def softmax(x, axis=-1):
    """
    Numerically stable softmax.
    
    Key insight: softmax(x) = softmax(x - max(x))
    This prevents overflow in exp(x) for large x.
    
    Args:
        x: np.array of any shape
        axis: axis along which to compute softmax
    """
    # Subtract max for numerical stability
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    
    # Normalize
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# Test cases
print("Test 1: Simple case")
logits = np.array([1.0, 2.0, 3.0])
probs = softmax(logits)
print(f"  Input: {logits}")
print(f"  Output: {probs}")
print(f"  Sum: {probs.sum():.4f} (should be 1.0)")

print("\nTest 2: Large values (potential overflow)")
logits_large = np.array([1000.0, 1001.0, 1002.0])
probs_large = softmax(logits_large)
print(f"  Input: {logits_large}")
print(f"  Output: {probs_large}")
print(f"  No overflow!")

print("\nTest 3: Batch of logits")
logits_batch = np.array([[1, 2, 3], [4, 5, 6]])
probs_batch = softmax(logits_batch, axis=1)
print(f"  Input shape: {logits_batch.shape}")
print(f"  Output:\n{probs_batch}")
print(f"  Row sums: {probs_batch.sum(axis=1)} (should be [1.0, 1.0])")

### Q7: Compute IoU (Intersection over Union)

**Interviewer**: "Given two bounding boxes, compute their IoU. Used in object detection."

In [ ]:
def compute_iou(box1, box2):
    """
    Compute Intersection over Union (IoU) of two bounding boxes.
    
    Args:
        box1, box2: [x1, y1, x2, y2] where (x1,y1) is top-left, (x2,y2) is bottom-right
    
    Returns:
        iou: float in [0, 1]
    """
    # Unpack coordinates
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    # Compute intersection
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    # Check if boxes intersect
    if inter_x_max < inter_x_min or inter_y_max < inter_y_min:
        return 0.0
    
    intersection = (inter_x_max - inter_x_min) * (inter_y_max - inter_y_min)
    
    # Compute areas
    area1 = (x1_max - x1_min) * (y1_max - y1_min)
    area2 = (x2_max - x2_min) * (y2_max - y2_min)
    
    # Compute union
    union = area1 + area2 - intersection
    
    # IoU
    iou = intersection / union if union > 0 else 0.0
    
    return iou

# Test cases
print("Test 1: Identical boxes")
box1 = [0, 0, 10, 10]
box2 = [0, 0, 10, 10]
print(f"  IoU: {compute_iou(box1, box2):.4f} (should be 1.0)")

print("\nTest 2: No overlap")
box1 = [0, 0, 10, 10]
box2 = [20, 20, 30, 30]
print(f"  IoU: {compute_iou(box1, box2):.4f} (should be 0.0)")

print("\nTest 3: Partial overlap")
box1 = [0, 0, 10, 10]
box2 = [5, 5, 15, 15]
iou = compute_iou(box1, box2)
print(f"  IoU: {iou:.4f}")
print(f"  (Intersection = 25, Union = 175, IoU = 25/175 = 0.1429)")

## Section 4: System Design Questions

### Q8: Design a Real-time Recommendation System

**Interviewer**: "Design a recommendation system for Netflix. Handle 200M users, <100ms latency."

**Answer Framework**:

```
1. REQUIREMENTS
   Functional:
   - Personalized recommendations
   - Real-time updates
   - Handle cold start
   
   Non-Functional:
   - Latency: P99 < 100ms
   - Scale: 200M users, 10k movies
   - Availability: 99.99%

2. HIGH-LEVEL ARCHITECTURE
   
   ┌──────────┐
   │  Client  │
   └─────┬────┘
         │
   ┌─────▼────────┐
   │  API Gateway │
   └─────┬────────┘
         │
   ┌─────▼──────────────────────────────┐
   │    Recommendation Service          │
   │  ┌─────────────────────────────┐   │
   │  │ 1. Candidate Generation     │   │ <-- Retrieve top 1000
   │  │    (Two-Tower Model)        │   │
   │  └─────────────┬───────────────┘   │
   │                │                    │
   │  ┌─────────────▼───────────────┐   │
   │  │ 2. Ranking                  │   │ <-- Rank top 50
   │  │    (Deep NN)                │   │
   │  └─────────────┬───────────────┘   │
   │                │                    │
   │  ┌─────────────▼───────────────┐   │
   │  │ 3. Business Logic           │   │ <-- Diversity, freshness
   │  └─────────────────────────────┘   │
   └────────────────────────────────────┘
         │              │
   ┌─────▼────┐   ┌────▼──────┐
   │  Redis   │   │ Feature   │
   │  Cache   │   │  Store    │
   └──────────┘   └───────────┘
   
3. DATA PIPELINE
   
   User Events → Kafka → Feature Engineering → Model Training
   (clicks, views)         (Spark Streaming)     (daily batch)
   
4. MODEL ARCHITECTURE
   
   Stage 1: Two-Tower (Candidate Generation)
   ┌──────────────┐     ┌──────────────┐
   │  User Tower  │     │  Item Tower  │
   │ (128-dim)    │     │ (128-dim)    │
   └──────┬───────┘     └──────┬───────┘
          │                    │
          └──────── ⊙ ─────────┘
                 (dot product)
   
   Stage 2: Ranking (Deep NN)
   [user features, item features, context] → MLP → P(click)

5. CACHING STRATEGY
   
   L1: User embeddings (Redis, TTL=1hr)
   L2: Item embeddings (Redis, TTL=24hr)
   L3: Pre-computed recommendations (Redis, TTL=10min)

6. SCALING
   
   - Horizontal: Shard users by ID
   - Model serving: vLLM / TensorRT
   - Database: Cassandra for user history
   - Vector DB: Faiss for ANN search

7. METRICS
   
   Online:
   - Click-through rate (CTR)
   - Watch time
   - Retention
   
   Offline:
   - AUC, NDCG@k
   - Diversity
   - Coverage

8. FAILURE MODES & MITIGATION
   
   | Failure | Mitigation |
   |---------|------------|
   | Model down | Fallback to popularity |
   | Cache miss | Compute on-the-fly |
   | Cold start | Content-based filtering |
   | Data skew | Debiasing techniques |
```

## Section 5: Probability & Statistics

### Q9: Explain and Implement Naive Bayes

**Mathematical Foundation**:

$$P(y|X) = \frac{P(X|y) P(y)}{P(X)}$$

For classification, we choose $\hat{y} = \arg\max_y P(y|X)$

**Naive assumption**: Features are independent given the class:
$$P(X|y) = \prod_{i=1}^{n} P(x_i|y)$$

In [ ]:
class NaiveBayesClassifier:
    """Gaussian Naive Bayes from scratch."""
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        # Initialize mean, variance, and priors
        self.mean = np.zeros((n_classes, n_features))
        self.var = np.zeros((n_classes, n_features))
        self.priors = np.zeros(n_classes)
        
        # Compute statistics for each class
        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.mean[idx, :] = X_c.mean(axis=0)
            self.var[idx, :] = X_c.var(axis=0)
            self.priors[idx] = X_c.shape[0] / n_samples
    
    def _gaussian_pdf(self, x, mean, var):
        """Gaussian probability density function."""
        eps = 1e-9  # Prevent division by zero
        coefficient = 1.0 / np.sqrt(2 * np.pi * var + eps)
        exponent = np.exp(-((x - mean) ** 2) / (2 * var + eps))
        return coefficient * exponent
    
    def predict(self, X):
        predictions = []
        
        for x in X:
            posteriors = []
            
            for idx, c in enumerate(self.classes):
                # Log probability to avoid underflow
                prior = np.log(self.priors[idx])
                likelihood = np.sum(np.log(
                    self._gaussian_pdf(x, self.mean[idx], self.var[idx])
                ))
                posterior = prior + likelihood
                posteriors.append(posterior)
            
            predictions.append(self.classes[np.argmax(posteriors)])
        
        return np.array(predictions)

# Test
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

nb = NaiveBayesClassifier()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Naive Bayes Accuracy: {accuracy:.4f}")

# Compare with sklearn
from sklearn.naive_bayes import GaussianNB
sklearn_nb = GaussianNB()
sklearn_nb.fit(X_train, y_train)
sklearn_pred = sklearn_nb.predict(X_test)
sklearn_acc = accuracy_score(y_test, sklearn_pred)

print(f"Sklearn Naive Bayes Accuracy: {sklearn_acc:.4f}")
print(f"\nOur implementation matches sklearn!")

## Section 6: Quick Fire Questions

### Common Interview Questions & Concise Answers

**Q**: What's the difference between L1 and L2 regularization?
- **L1 (Lasso)**: $\lambda \sum |w_i|$ → Sparse solutions (feature selection)
- **L2 (Ridge)**: $\lambda \sum w_i^2$ → Smooth solutions (shrink all weights)

**Q**: Explain vanishing gradients.
- **Problem**: In deep networks, gradients shrink exponentially through layers
- **Cause**: Sigmoid derivatives max out at 0.25
- **Solutions**: ReLU, Batch Norm, Skip connections, Better initialization

**Q**: Why use cross-entropy loss instead of MSE for classification?
- **Cross-entropy**: Measures distribution difference, gradient = (p - y)
- **MSE**: Gradient vanishes when prediction is very wrong

**Q**: What is the attention mechanism?
- **Purpose**: Dynamically focus on relevant parts of input
- **Formula**: $\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$
- **Benefit**: Handles long sequences better than RNNs

**Q**: Explain the transformer architecture in 30 seconds.
- **Core**: Self-attention + Feed-forward, no recurrence
- **Encoder**: Process input
- **Decoder**: Generate output autoregressively
- **Key innovation**: Parallel processing of sequences

**Q**: How do you handle class imbalance?
1. **Resampling**: Oversample minority, undersample majority
2. **Weighted loss**: Increase weight for minority class
3. **Evaluation**: Use F1, precision-recall, not accuracy
4. **SMOTE**: Synthetic minority samples

**Q**: Describe a training pipeline for a production ML model.
```
Data Collection → Feature Engineering → Train/Val Split →
Model Training → Hyperparameter Tuning → Evaluation →
Model Registry → Deployment → Monitoring → Retraining
```

## Practice Plan

### Week 1: Fundamentals
- [ ] Review bias-variance, cross-validation
- [ ] Implement optimizers from scratch
- [ ] Practice gradient checking

### Week 2: Deep Learning
- [ ] Implement backpropagation
- [ ] Code attention mechanism
- [ ] Build transformer from scratch

### Week 3: System Design
- [ ] Design recommendation system
- [ ] Design search ranking
- [ ] Design model serving pipeline

### Week 4: Mock Interviews
- [ ] Leetcode ML problems
- [ ] System design practice
- [ ] Behavioral questions

## Resources

- **Books**: Deep Learning (Goodfellow), ML Interview Book (Chip Huyen)
- **Coding**: LeetCode ML, Pramp
- **System Design**: System Design Interview, ML System Design
- **Mock Interviews**: interviewing.io, Pramp

Good luck! 🚀